# Arreglos, dimensiones y selección con NumPy

Representaremos rondas de temperatura de tres salas. Antes de transformar los
valores, estableceremos qué significa cada posición del arreglo.

## Contenido

1. Listas y arreglos.
2. Dimensiones, forma y tamaño.
3. Tipos numéricos.
4. Selección de filas y columnas.
5. Máscaras booleanas.
6. Vistas y copias.
7. Resúmenes por eje.
8. Mediciones incompletas y paso a una aplicación.

Abre esta notebook desde `01-exploracion` con `uv run --locked jupyter lab`.
Los ejercicios tienen celdas independientes: puedes recorrer todos los ejemplos
sin completarlos. Cada ejercicio se resuelve en su celda y conserva sus comprobaciones.

In [ ]:
import sys
from pathlib import Path

import numpy as np

print(f"Python: {sys.executable}")
print(f"Working directory: {Path.cwd()}")
print(f"NumPy: {np.__version__}")

data_dir = Path.cwd().parent / "datos"
if not (data_dir / "readings.csv").is_file():
    raise FileNotFoundError(
        "Start the notebook from the 01-exploracion project directory"
    )

## 1. Listas y arreglos

NumPy ofrece `ndarray`, un arreglo de elementos con un tipo común. Su utilidad
no se limita a escribir menos ciclos: también hace explícita la organización de
los datos. Una lista sigue siendo útil cuando necesitamos una colección flexible.

Predice qué devuelve cada multiplicación: una opera sobre una lista y la otra
sobre sus valores numéricos.

[Introducción oficial a NumPy](https://numpy.org/doc/stable/user/absolute_beginners.html).

In [ ]:
temperature_list = [22.0, 24.0, 20.0]
temperature_array = np.array(temperature_list)
print(temperature_list * 2)
print(temperature_array * 2)

El arreglo también puede construirse con `zeros`, `ones`, `arange` o `linspace`.
`arange` usa un paso y excluye el límite superior; `linspace` fija una cantidad de
puntos y, por defecto, incluye ambos extremos. Estas secuencias son ejemplos de
creación, no las mediciones del caso.

In [ ]:
print(np.zeros((2, 3)))
print(np.ones(3))
print(np.arange(0, 8, 2))
print(np.linspace(20, 30, 5))

## 2. Dimensiones, forma y tamaño

Usaremos este orden de columnas: sala norte, sala sur y almacén. Cada fila
corresponde a una ronda en la que se midieron las tres salas.

| Ronda | `north_room` | `south_room` | `storage_room` |
|---|---:|---:|---:|
| 0 | 22 | 24 | 20 |
| 1 | 23 | 25 | 21 |
| 2 | 24 | 26 | 22 |

`loadtxt` lee valores numéricos; `skiprows=1` omite el encabezado y `ndmin=2`
mantiene al menos dos dimensiones, incluso para un archivo de una sola fila.
El orden de los nombres se conserva por acuerdo con el archivo, no dentro del
arreglo. La [ficha de datos](../datos/README.md) describe las unidades y su origen.

[Referencia de loadtxt](https://numpy.org/doc/stable/reference/generated/numpy.loadtxt.html).

In [ ]:
room_names = ("north_room", "south_room", "storage_room")
readings = np.loadtxt(data_dir / "readings.csv", delimiter=",", skiprows=1, ndmin=2)
print(readings)

`ndim` cuenta ejes; `shape` indica cuántas posiciones hay en cada eje; `size`
es el total de elementos. `nbytes` cuenta los bytes del búfer de datos, no todos
los objetos ni el consumo total de Python.

```text
                        axis 1: salas →
                      norte   sur   almacén
axis 0: rondas ↓    0   22     24      20
                   1   23     25      21
                   2   24     26      22
                   …
```

[Propiedades de ndarray](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.html).

In [ ]:
print(f"Dimensions: {readings.ndim}")
print(f"Shape: {readings.shape}")
print(f"Elements: {readings.size}")
print(f"Data type: {readings.dtype}")
print(f"Bytes per element: {readings.itemsize}")
print(f"Data bytes: {readings.nbytes}")
assert readings.size == 8 * 3

In [ ]:
scalar = np.array(22.0)
vector = np.array([22.0, 24.0, 20.0])
matrix = np.array([[22.0, 24.0, 20.0]])
for name, values in [("scalar", scalar), ("vector", vector), ("matrix", matrix)]:
    print(name, values.ndim, values.shape, values.size)

### Ejercicio 1: una ronda, dos representaciones

Construye `one_round` como vector de tres temperaturas y `one_round_table`
como matriz de una fila. Comprueba las formas `(3,)` y `(1, 3)`. Explica por qué
ambos tienen tres elementos y distinto número de ejes.

Anticipa el resultado antes de ejecutar. Anota después qué comprobaste.

In [ ]:
# Write your solution here.

## 3. Tipos numéricos

El tipo del arreglo determina cómo se representa cada elemento. `int64` guarda
enteros; `float64` permite decimales y valores especiales como `nan`.
Mezclar números con texto puede producir un arreglo de cadenas: crear el arreglo
no garantiza que pueda usarse en operaciones numéricas.

La asignación siguiente no convierte automáticamente el arreglo entero en uno
flotante. Observa qué sucede con la fracción. Para preservarla, crea primero un
arreglo flotante o usa `astype` antes de asignar.

[Tipos de datos de NumPy](https://numpy.org/doc/stable/user/basics.types.html).

In [ ]:
integer_readings = np.array([22, 24, 20], dtype=np.int64)
integer_readings[0] = 22.75
print(integer_readings)

float_readings = np.array([22, 24, 20], dtype=np.float64)
float_readings[0] = 22.75
print(float_readings)
print(np.array([22, "missing"]).dtype)

`float32` y `float64` requieren diferente espacio por elemento. Menos bytes no
implica que un tipo sea adecuado: también importa la precisión que necesita la
operación. Cambiar a float después de perder una fracción no la recupera.

In [ ]:
readings32 = readings.astype(np.float32)
print(readings.nbytes, readings32.nbytes)
print(integer_readings.astype(np.float64))

### Ejercicio 2: preservar una temperatura decimal

Crea un arreglo con `22`, `24` y `20` que permita sustituir el primer valor por
`22.75` sin truncarlo. Comprueba el valor con `np.testing.assert_allclose`.

Anticipa el resultado antes de ejecutar. Anota después qué comprobaste.

In [ ]:
# Write your solution here.

## 4. Selección de filas y columnas

En `readings[row, column]`, el primer índice elige la ronda y el segundo la sala.
El slice excluye su límite superior. Un índice entero elimina ese eje del
resultado; un slice lo conserva.

[Selección de elementos y slicing](https://numpy.org/doc/stable/user/basics.indexing.html).

In [ ]:
print(readings[0, 1])
print(readings[-1])
print(readings[:3, :2])
print(readings[:, 0])
print(readings[:, 0:1])
print(readings[:, 0].shape, readings[:, 0:1].shape)

Una lista de índices permite escoger columnas no consecutivas. Comprueba el
orden resultante: `[2, 0]` coloca primero el almacén y después la sala norte.

In [ ]:
selected_rooms = readings[:, [2, 0]]
print(selected_rooms[:2])
print(selected_rooms.shape)

### Ejercicio 3: últimas rondas

Obtén las últimas tres rondas para sala norte y almacén, en ese orden. El
resultado debe ser `[[27, 25], [28, 26], [29, 27]]`, de forma `(3, 2)`.
Comprueba valores y forma.

Anticipa el resultado antes de ejecutar. Anota después qué comprobaste.

In [ ]:
# Write your solution here.

### Ejercicio 4: una columna con dos ejes

Selecciona la sala sur conservando la forma `(8, 1)`. Contrasta la expresión
con una selección que devuelva `(8,)`. Explica qué eje desapareció.

Anticipa el resultado antes de ejecutar. Anota después qué comprobaste.

In [ ]:
# Write your solution here.

## 5. Máscaras booleanas

Una comparación produce un booleano por elemento. Para seleccionar rondas usando
una sala, la máscara debe tener una posición por fila. `&` combina condiciones
por elemento; cada comparación necesita paréntesis.

In [ ]:
north_temperatures = readings[:, 0]
warm_rounds = north_temperatures >= 27
print(warm_rounds)
print(warm_rounds.shape)
print(readings[warm_rounds])

comfortable_rounds = (north_temperatures >= 24) & (north_temperatures < 27)
print(readings[comfortable_rounds])

Una máscara del mismo tamaño que toda la matriz selecciona celdas y devuelve
un vector. Ya no conserva una fila por ronda. Compara ambas selecciones antes de
usar el resultado en un resumen.

In [ ]:
cell_mask = readings >= 28
print(cell_mask.shape)
print(readings[cell_mask])
print(readings[cell_mask].shape)
print(readings[readings[:, 1] >= 28].shape)

### Ejercicio 5: un intervalo

Selecciona las rondas cuya temperatura de la sala sur esté entre `26` incluido
y `29` excluido. Deben quedar tres filas completas. Escribe la máscara, imprime
su forma y comprueba las temperaturas `[26, 27, 28]` de esa columna.

Anticipa el resultado antes de ejecutar. Anota después qué comprobaste.

In [ ]:
# Write your solution here.

## 6. Vistas y copias

Un slice básico produce una vista del mismo búfer. Modificar esa vista puede
cambiar el arreglo de origen. `.copy()` crea datos independientes; la selección
con índices enteros o booleanos también obtiene una copia.

Trabajaremos sobre `working_readings` para que el experimento no cambie
`readings`. Observa la diferencia entre alias de datos y archivos: ninguna de
estas asignaciones escribe el CSV.

[Copias y vistas](https://numpy.org/doc/stable/user/basics.copies.html).

In [ ]:
working_readings = readings.copy()
north_view = working_readings[:, 0]
north_copy = working_readings[:, 0].copy()
print(np.shares_memory(working_readings, north_view))
print(np.shares_memory(working_readings, north_copy))
north_view[0] = -10
print(working_readings[0, 0], north_copy[0], readings[0, 0])

In [ ]:
warm_copy = readings[readings[:, 0] >= 27]
warm_copy[0, 0] = -20
print(np.shares_memory(readings, warm_copy))
print(readings[5, 0])

edited_readings = readings.copy()
edited_readings[edited_readings[:, 0] >= 27, 0] = 27
print(edited_readings[:, 0])

Obtener una selección con máscara crea una copia, pero asignar directamente
con `array[mask] = value` modifica el arreglo destino. La distinción está en qué
objeto recibe la asignación.

### Ejercicio 6: una selección independiente

Crea `last_rounds` con las dos últimas rondas, cambia su primera temperatura
a `100` y comprueba que `readings` sigue igual que antes. Usa `copy()` y
`np.testing.assert_array_equal` para demostrarlo.

Anticipa el resultado antes de ejecutar. Anota después qué comprobaste.

In [ ]:
# Write your solution here.

## 7. Resúmenes por eje

Una reducción elimina el eje sobre el que opera. `axis=0` combina las rondas y
conserva las tres salas; `axis=1` combina las salas y conserva las ocho rondas.
Sin `axis`, la media resume todas las celdas en un valor.

Aquí todas las columnas tienen la misma unidad. Una media entre columnas con
unidades distintas, como temperatura y humedad, no tendría esa interpretación.

[Promedios con numpy.mean](https://numpy.org/doc/stable/reference/generated/numpy.mean.html).

In [ ]:
room_means = readings.mean(axis=0)
round_means = readings.mean(axis=1)
print(room_means, room_means.shape)
print(round_means, round_means.shape)
print(readings.mean())
np.testing.assert_allclose(room_means, [25.5, 27.5, 23.5])

### Ejercicio 7: máximos y medias

Obtén el máximo de cada sala y la media de cada ronda. Comprueba los máximos
`[29, 31, 27]` y las formas `(3,)` y `(8,)`. Describe el eje que se reduce en
cada caso.

Anticipa el resultado antes de ejecutar. Anota después qué comprobaste.

In [ ]:
# Write your solution here.

## 8. Mediciones incompletas

`isfinite` identifica valores distintos de `nan` y los infinitos. `all(axis=1)`
exige que las tres posiciones de una ronda sean finitas. Usaremos esa política
para conservar las mismas rondas en todas las salas.

Es una regla del ejercicio: eliminar una fila también descarta mediciones válidas
de sus otras salas. La decisión debe quedar visible en el conteo de rechazos.

[isfinite](https://numpy.org/doc/stable/reference/generated/numpy.isfinite.html) ·
[all](https://numpy.org/doc/stable/reference/generated/numpy.all.html).

In [ ]:
quality_readings = np.loadtxt(
    data_dir / "readings_quality.csv", delimiter=",", skiprows=1, ndmin=2
)
finite_cells = np.isfinite(quality_readings)
complete_rows = finite_cells.all(axis=1)
print(finite_cells)
print(complete_rows)
print(quality_readings[complete_rows])

### Ejercicio 8: contar los rechazos

Cuenta las filas aceptadas y rechazadas de `quality_readings`. Comprueba que
son dos y dos, y que las medias del arreglo seleccionado son `[23, 25, 21]`.
¿Qué pasaría si no quedara ninguna fila completa?

Anticipa el resultado antes de ejecutar. Anota después qué comprobaste.

In [ ]:
# Write your solution here.

## Continuación: llevar las operaciones al proyecto

En el proyecto `02-reporte-mediciones`, estas operaciones forman funciones que
se importan desde una notebook y desde `main.py`. Sigue la [guía](../GUIA.md) para
cambiar de proyecto y abrir `u2_n2_reporte_mediciones.ipynb`.

Antes de continuar, reinicia el kernel y ejecuta todas las celdas. Los resultados
no deben depender de haber ejecutado una celda fuera de orden.